[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_enrichment.ipynb)

# DataSage Stage 2: Enrichment GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.

**Fully self-contained** — runs in Colab with no repo clone or project imports.

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY`.

In [1]:
# Setup: install dependencies
!pip install -q unsloth trl datasets wandb requests vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 148.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

Loaded keys from Colab Secrets
Keys loaded.


In [4]:
# Config + parser + reward helpers (all inlined, no project imports)
import json
import re
import requests

# ---- Config constants ----
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-enrichment.hf.space"
HF_REPO = "ricalanis/datasage-qwen-enrichment"

LEARNING_RATE = 5e-6
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
NUM_GENERATIONS = 8
MAX_COMPLETION_LENGTH = 256
MAX_PROMPT_LENGTH = 1024


# ---- Completion text extraction ----
def _get_text(completion) -> str:
    """Extract text from a completion (str or chat message list)."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        # Chat format: [{"role": "assistant", "content": "..."}]
        return completion[-1]["content"] if completion else ""
    return str(completion)


# ---- Parser ----
def parse_enrichment_action(text: str) -> dict:
    """Parse model output into an enrichment action dict."""
    json_match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "field_name" in data:
                return data
        except json.JSONDecodeError:
            pass

    text_lower = text.lower()
    sources = [
        "salary_band", "tenure_risk", "satisfaction_index", "industry_benchmark",
        "flight_risk_score", "deal_size_category", "velocity_score",
        "win_probability_model", "industry_code", "competitive_risk",
        "schedule_risk_score", "resource_utilization", "dependency_chain_depth",
        "burndown_rate", "delay_probability", "sla_compliance_flag", "mttr_band",
        "escalation_path", "incident_severity_score", "recurring_pattern_flag",
    ]
    for source in sources:
        if source.replace("_", " ") in text_lower or source in text_lower:
            return {"operation": "add_field", "field_name": source, "source": source,
                    "logic": "", "params": {}}

    return {"operation": "add_field", "field_name": "unknown", "source": "", "logic": "", "params": {}}


# ---- Reward helpers ----
_ACTION_JSON_RE = re.compile(r'\{[^{}]*"operation"[^{}]*\}')
ENRICHMENT_VALID_OPS = {"add_field", "lookup", "compute_derived", "add_category"}


def source_relevance_reward(completions, **kwargs) -> list[float]:
    """Reward picking a valid enrichment source for the domain."""
    sources_list = kwargs.get("available_sources", [])
    if not sources_list:
        return [0.0] * len(completions)
    rewards = []
    for completion, available in zip(completions, sources_list):
        text = _get_text(completion)
        action_dict = parse_enrichment_action(text)
        source = action_dict.get("source", "")
        if source in available:
            rewards.append(1.0)
        elif action_dict.get("field_name", "unknown") != "unknown":
            rewards.append(0.3)
        else:
            rewards.append(0.0)
    return rewards


def enrichment_json_format_reward(completions, **kwargs) -> list[float]:
    """Reward well-formed JSON enrichment actions."""
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        match = _ACTION_JSON_RE.search(text)
        if match:
            try:
                data = json.loads(match.group())
                op_ok = data.get("operation") in ENRICHMENT_VALID_OPS
                field_ok = "field_name" in data and data["field_name"] != "unknown"
                if op_ok and field_ok:
                    rewards.append(1.0)
                elif op_ok:
                    rewards.append(0.5)
                else:
                    rewards.append(0.2)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


print(f"Environment URL: {ENV_URL}")
print(f"Base model: {BASE_MODEL}")

Environment URL: https://ricalanis-datasage-enrichment.hf.space
Base model: unsloth/Qwen2.5-3B-Instruct


In [5]:
# Model loading via Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True,
    fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded: {BASE_MODEL}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-08 10:18:20 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 59.17%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.49 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 80.
Unsloth: vLLM's KV Cache can use up to 

/usr/local/lib/python3.12/dist-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


WARNING 03-08 10:18:35 [arg_utils.py:1321] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 03-08 10:19:01 [model.py:531] Resolved architecture: Qwen2ForCausalLM
INFO 03-08 10:19:01 [model.py:1554] Using max model len 2048
INFO 03-08 10:19:01 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'bfloat16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.2.mlp', 'model.layers.3.mlp', 'model.layers.30.mlp'], 'llm_int8_threshold': 6.0}
INFO 03-08 10:19:01 [vllm.py:747] Asynchronous scheduli

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded: unsloth/Qwen2.5-3B-Instruct


In [6]:
# System prompt and task descriptions
SYSTEM_PROMPT = """\
You are a data enrichment agent. You enrich enterprise datasets by adding \
derived fields, lookups, and computed columns across multiple domains \
(HR, Sales, Project Management, IT Operations).

Analyze the schema and available sources below, then respond with a JSON \
enrichment action:
{"operation": "<op>", "field_name": "<name>", "source": "<source>", "logic": "<logic>", "params": {}}

Available operations:
- add_field: Add a new enrichment field from a known source
- lookup: Look up external reference data
- compute_derived: Compute a derived metric from existing columns
- add_category: Add a categorical classification

Identify the most valuable enrichment to add and act."""

TASK_DESCRIPTIONS = [
    "Enrich this dataset by adding the most valuable derived field.",
    "Add an enrichment that increases analytical coverage the most.",
    "Look at the available sources and add the most impactful one.",
    "This dataset needs enrichment. Choose the best source to add.",
    "Maximize enrichment coverage by adding the most useful field.",
    "Analyze the schema and pick the enrichment with highest value.",
    "Add a derived field that enables the most downstream analysis.",
    "Choose an enrichment source that fills the biggest analytics gap.",
]

In [7]:
# Dataset: pre-built with real environment observations
import random
from datasets import Dataset

random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []
available_sources_list = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    resp = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed})
    resp.raise_for_status()
    obs = resp.json()["observation"]

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs['domain']}\n\nSchema:\n{obs['schema_info']}\n\n"
            f"Available Enrichment Sources: {', '.join(obs['available_sources'])}\n\n"
            f"Possible Enrichments: {', '.join(obs['possible_enrichments'])}\n\n"
            f"Data Preview:\n{obs['data_preview']}\n\nTask: {task_desc}"
        )},
    ])
    available_sources_list.append(obs["available_sources"])
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
    "available_sources": available_sources_list,
})
print(f"Dataset size: {len(dataset)}")

Building dataset with real environment observations...
  Built 16/64 examples
  Built 32/64 examples
  Built 48/64 examples
  Built 64/64 examples
Dataset size: 64


In [8]:
# Environment reward function (calls env directly with stored seeds)
def enrichment_env_reward(completions, **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for completion, seed in zip(completions, seeds):
        try:
            text = _get_text(completion)
            action_dict = parse_enrichment_action(text)
            requests.post(f"{ENV_URL}/web/reset", json={"seed": int(seed)})
            resp = requests.post(f"{ENV_URL}/web/step", json={"action": action_dict})
            resp.raise_for_status()
            rewards.append(float(resp.json().get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [9]:
# GRPO training config
from trl import GRPOConfig

training_args = GRPOConfig(
    output_dir="./outputs/enrichment-grpo",
    use_vllm=True,
    learning_rate=LEARNING_RATE,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_grad_norm=0.1,
    logging_steps=1, save_steps=50,
    report_to="wandb", run_name="datasage-enrichment-grpo-v2",
)
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


'datasage'

In [10]:
# Create trainer and train
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[enrichment_env_reward, source_relevance_reward, enrichment_json_format_reward],
)
print("Starting Stage 2 (Enrichment) GRPO training...")
trainer.train()

Starting Stage 2 (Enrichment) GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 64 | Num Epochs = 3 | Total steps = 192
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ricalanis to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


WARNING 03-08 10:24:01 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / enrichment_env_reward / mean,rewards / enrichment_env_reward / std,rewards / source_relevance_reward / mean,rewards / source_relevance_reward / std,rewards / enrichment_json_format_reward / mean,rewards / enrichment_json_format_reward / std
1,0.132800,1.112500,0.035355,67.375000,42.000000,107.000000,0.000000,67.375000,42.000000,107.000000,0,0,0,0,0,-0.000000,0.112500,0.035355,1.000000,0.000000,0.000000,0.000000
2,-0.029500,0.862500,0.471888,78.500000,55.000000,127.000000,0.000000,78.500000,55.000000,127.000000,No Log,No Log,No Log,No Log,No Log,-0.000000,0.112500,0.035355,0.750000,0.462910,0.000000,0.000000
3,0.157200,1.112500,0.035355,83.000000,46.000000,181.000000,0.000000,83.000000,46.000000,181.000000,No Log,No Log,No Log,No Log,No Log,0.000378,0.112500,0.035355,1.000000,0.000000,0.000000,0.000000
4,-0.065800,1.000000,0.366450,85.500000,35.000000,206.000000,0.000000,85.500000,35.000000,206.000000,No Log,No Log,No Log,No Log,No Log,0.000764,0.125000,0.046291,0.875000,0.353553,0.000000,0.000000
5,-0.262200,1.125000,0.046291,92.500000,56.000000,191.000000,0.000000,92.500000,56.000000,191.000000,No Log,No Log,No Log,No Log,No Log,0.000825,0.125000,0.046291,1.000000,0.000000,0.000000,0.000000
6,0.034100,1.150000,0.053452,99.125000,57.000000,158.000000,0.000000,99.125000,57.000000,158.000000,No Log,No Log,No Log,No Log,No Log,0.000836,0.150000,0.053452,1.000000,0.000000,0.000000,0.000000
7,-0.175400,1.150000,0.053452,57.875000,40.000000,88.000000,0.000000,57.875000,40.000000,88.000000,No Log,No Log,No Log,No Log,No Log,0.000840,0.150000,0.053452,1.000000,0.000000,0.000000,0.000000
8,-0.236400,1.150000,0.053452,87.375000,37.000000,256.000000,0.125000,63.285717,37.000000,88.000000,No Log,No Log,No Log,No Log,No Log,0.000867,0.150000,0.053452,1.000000,0.000000,0.000000,0.000000
9,-0.025600,1.000000,0.366450,85.125000,48.000000,167.000000,0.000000,85.125000,48.000000,167.000000,No Log,No Log,No Log,No Log,No Log,0.000905,0.125000,0.046291,0.875000,0.353553,0.000000,0.000000
10,0.015900,1.125000,0.046291,68.000000,44.000000,102.000000,0.000000,68.000000,44.000000,102.000000,No Log,No Log,No Log,No Log,No Log,0.000911,0.125000,0.046291,1.000000,0.000000,0.000000,0.000000


profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▃▆▇▆▇▇▃▇▅▄▁▅▅▄▇█▆▄▃▅▃▅▅▃▄▄▃▅▃▂▅▇▅██▇█▅▆▄
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▁▇▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
profiling/Time taken: UnslothGRPOTrainer.enrichment_env_reward,▄▅▅▅▆▃▆▄▁▁▅▃█▃▅▂▅▃▃▃▄▂▃▂▄▃▃▃▂▄▄▃▄█▅▅▄▄▆▄
profiling/Time taken: UnslothGRPOTrainer.enrichment_json_format_reward,▃▃▄▃▄▅▅▃▆▃▃▅▅▂▁▄▄▂▃▄▃▃▆█▄▇▁▅▅▁▅▆▂▂▅▄▅▅▃▃
profiling/Time taken: UnslothGRPOTrainer.source_relevance_reward,▃▇▂▄▄▄▆▄▄▇▂▅█▅▅▄▅█▃▃▃▂▇▇▆▆▇▁▃▄▅▅▂▅▄▅▅▂▄▆
profiling/Time taken: UnslothGRPOTrainer.vLLM.generate,▅▃▃▁▇▃▃▃▃▄▃▄▅▅▅▃▃▇▃▇▂▂▄█▃▆▄▃▃▂▆▂▇▁▄▁▃▁▃▄
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_min,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+25,...


TrainOutput(global_step=192, training_loss=-0.02330715726054411, metrics={'train_runtime': 3958.999, 'train_samples_per_second': 0.048, 'train_steps_per_second': 0.048, 'total_flos': 0.0, 'train_loss': -0.02330715726054411})

In [11]:
# Save and push to Hub
output_dir = "./outputs/enrichment-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

print(f"Pushing to Hub: {HF_REPO}")
trainer.push_to_hub(HF_REPO)
print(f"Model pushed to https://huggingface.co/{HF_REPO}")

Model saved to ./outputs/enrichment-grpo-final
Pushing to Hub: ricalanis/datasage-qwen-enrichment


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  535kB /  120MB            

  ...hment-grpo/tokenizer.json:   0%|          | 27.8kB / 11.4MB            

  ...nt-grpo/training_args.bin:   1%|          |  72.0B / 7.38kB            

Model pushed to https://huggingface.co/ricalanis/datasage-qwen-enrichment
